**[CHANGE]**: Load and clean facilities list

***

__author__ = "Macarena Merida Floriano, mmef" <br>
__maintainer__ = "Macarena Merida Floriano, mmef"<br>
__email__ = "mmef@gmv.com"

***

In [ ]:
import pandas as pd

In [ ]:
data_path = "../data/HSF MASTER FACILITY LIST.xlsx"
fac_df = pd.read_excel(data_path)
fac_df.head(10) 

,State,State_code,County,County_code,Payam,Payam_code,Health Facility Name,latitude,longitude,Functional Status
0,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,abiemnom phcc,9.398740,28.823400,Moderately Functional
1,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,awarpiny phcu,9.473620,28.888280,Moderately Functional
2,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,manajoga phcu,9.392128,28.811239,Moderately Functional
3,unity,SS06,abiemnhom,SS0601,abiemnhom,SS060101,panyang phcu,9.409190,28.912590,Moderately Functional
4,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,abyei civil hospital,9.592052,28.436265,Moderately Functional
5,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,abyei phcc,9.605027,28.432163,Moderately Functional
6,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,agany tok phcu,9.527582,28.434455,Not Functional
7,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,agok phcc,9.357027,28.582675,Not Functional
8,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,amenth bek hospital,9.584825,28.459247,Moderately Functional
9,abyei region,SS00,abyei region,SS0001,abyei region,SS000101,amiet phcu,9.715699,28.465735,Not Functional


In [ ]:
print("Original CSV facilities number:", fac_df.shape[0])
print("Original CSV columns:", fac_df.columns.tolist())

(1988, 10)

In [ ]:
fac_df_clean = fac_df.dropna(subset=['latitude', 'longitude', 'Functional Status', 'Health Facility Name', 'County_code', 'Payam_code'])
print("Cleaned CSV facilities number:", fac_df_clean.shape[0])

(1791, 10)

In [ ]:
print("Unique Functional Status values:", fac_df_clean['Functional Status'].unique())

['Moderately Functional' 'Not Functional' 'Highly Functional'
 'Limitedly Functional']


In [ ]:
# Select only functional facilities
fac_df_func = fac_df_clean[fac_df_clean['Functional Status'].str.strip().str.lower() != 'not functional']
print("Functional facilities number:", fac_df_func.shape[0])

(1480, 10)

In [12]:
# Check unique facilities based on coordinates
fac_df_func['coord_tuple'] = list(zip(fac_df_func['latitude'], fac_df_func['longitude']))
unique_coords = fac_df_func['coord_tuple'].nunique()
print(f"Number of unique facilities based on coordinates: {unique_coords}")
# print repeated coordinates
repeated_coords = fac_df_func[fac_df_func.duplicated(subset=['coord_tuple'], keep=False)]
print("Repeated coordinates:")
print(repeated_coords[['Health Facility Name', "Payam_code", 'latitude', 'longitude']])

Number of unique facilities based on coordinates: 1477
Repeated coordinates:
    Health Facility Name Payam_code  latitude  longitude
391          chuoke phcu   SS060201  9.144742  29.987039
392            guit phcc   SS060201  9.144742  29.987039
886           dangu phcu   SS100702  6.473300  27.734560
891        namutina phcc   SS100702  6.473300  27.734560
934          pachar phcu   SS060706  7.655025  30.247661
935         pathiel phcu   SS060702  7.655025  30.247661


/tmp/ipykernel_69748/953096928.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fac_df_func['coord_tuple'] = list(zip(fac_df_func['latitude'], fac_df_func['longitude']))


In [ ]:
# Drop duplicated facilities based on coordinates, keeping the first occurrence
fac_df_func_unique = fac_df_func.drop_duplicates(subset=['coord_tuple'], keep='first')
print("Functional facilities with unique coordinates number:", fac_df_func_unique.shape[0])

(1477, 11)

Save coordinates to check in QGIS

In [ ]:
coords = fac_df_func_unique[['County_code', 'Payam_code', 'Health Facility Name', 'longitude', 'latitude','Functional Status']] #.astype(float)
names = ["County_code", "Payam_code", "Health Facility Name", "longitude", "latitude", "Functional Status"]
coords_csv_path = "../data/valid_facilities.csv"
coords.to_csv(coords_csv_path, index=False, header=names,float_format='%.6f')